In [4]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.7 MB/s eta 0:00:00


In [1]:

from mpi4py import MPI
import tensorflow as tf
import numpy as np



comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# ---------------- Load Dataset ----------------

mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize dataset
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape for CNN input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# ---------------- Split Dataset ----------------

n = len(x_train)
chunk_size = n // size

start = rank * chunk_size

# Last process handles remaining data
if rank == size - 1:
    end = n
else:
    end = (rank + 1) * chunk_size

x_chunk = x_train[start:end]
y_chunk = y_train[start:end]

# ---------------- Define CNN Model ----------------

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),

    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        activation='relu'
    ),

    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        10,
        activation='softmax'
    )
])

# ---------------- Compile Model ----------------

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------- Training ----------------

epochs = 5

for epoch in range(epochs):

    # Train on local chunk
    model.fit(
        x_chunk,
        y_chunk,
        epochs=1,
        batch_size=32,
        verbose=0
    )

    # Evaluate training accuracy on local chunk
    train_loss, train_acc = model.evaluate(
        x_chunk,
        y_chunk,
        verbose=0
    )

    # Evaluate testing accuracy
    test_loss, test_acc = model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

    # Collect average accuracy from all processes
    global_train_acc = comm.allreduce(train_acc, op=MPI.SUM) / size
    global_test_acc = comm.allreduce(test_acc, op=MPI.SUM) / size

    # Print results only from master process
    if rank == 0:
        print(f"Epoch {epoch + 1}: "
              f"Train Accuracy = {global_train_acc:.4f}, "
              f"Test Accuracy = {global_test_acc:.4f}")

# ---------------- Final Message ----------------

if rank == 0:
    print("\nTraining Completed Successfully")



11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1: Train Accuracy = 0.9839, Test Accuracy = 0.9785
Epoch 2: Train Accuracy = 0.9905, Test Accuracy = 0.9832
Epoch 3: Train Accuracy = 0.9940, Test Accuracy = 0.9859
Epoch 4: Train Accuracy = 0.9964, Test Accuracy = 0.9861
Epoch 5: Train Accuracy = 0.9942, Test Accuracy = 0.9842

Training Completed Successfully
